In [128]:
import os
import pandas as pd
import re
import torch
from collections import defaultdict
from pathlib import Path
import numpy as np

def get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.add(c[:end_idx])
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw=[]
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
        raw.extend(concepts)
    
    return unit_concepts, set(raw)

def build_binary_mask(neuron_mask, foundational_concept_list) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(foundational_concept_list)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(foundational_concept_list):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor



def get_neurons_for_cps(concepts, mapping):
    neurons = []
    for neuron, cps in mapping.items():
        for c in cps:
            if c in concepts:
                neurons.append(neuron)
                break
    return neurons

def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(set)
    s=set()
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[unit].update(get_indiv_concepts(formula))
        
     
        for u, c in concept_dict.items():
            s.update(c)
    return concept_dict

In [132]:
def get_foundationals(root_dir):
    root_path = Path(root_dir)

    # Find all matching CSV files
    fldr_pattern = '*%Pruned'
    fldr_files = list(root_path.rglob(fldr_pattern))
    cross_iter_concept_dict=defaultdict(list)
    for fldr_file in fldr_files:
        concepts = []
        for csvs in os.listdir(os.path.join(root_dir, fldr_file)):

            if 'IOUS1024N' not in csvs: continue

            csv_file = os.path.join(root_dir, fldr_file, csvs)
            df = pd.read_csv(csv_file)
            for unit, formula in zip(df.unit, df.best_name):
                concepts.extend(get_indiv_concepts(formula))
        cross_iter_concept_dict[fldr_file]=set(concepts)
    preserved_concepts = set.intersection(*cross_iter_concept_dict.values())

    return list(preserved_concepts)

def get_topk_concepts(mask,k, concept_list):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    sorted_idx = np.argsort(col_sums)[::-1]
    top=[]
    freqs={}
    
    for i in range(len(sorted_idx)):
        idx = sorted_idx[i]
        top.append(concept_list[idx])
        freqs[concept_list[idx]]=col_sums[idx]
    return top, sum(list(freqs.values()))/len(freqs), freqs
root_dir='/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_5/Expls'
foundationals = get_foundationals(root_dir)
for direct in sorted(os.listdir(root_dir)):
    neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_5/Expls/{direct}')
    mask = build_binary_mask(neuron_formulas, foundationals)

    top,f,freqs=get_topk_concepts(mask,len(foundationals), foundationals)
    print(direct,f)
    print(freqs)


'''
'pre:tag:nn': 539.0,
 'hyp:tag:nn': 327.0,
 'oth:overlap:overlap25': 307.0,
 'oth:overlap:overlap50': 202.0,
 'pre:tag:jj': 178.0,
 'hyp:tok:outside': 149.0,
 'hyp:tok:sleeping': 145.0,
 'hyp:tag:in': 134.0,
 'hyp:tag:vbg': 133.0,
 'hyp:tag:dt': 133.0,
 'pre:tag:.': 133.0,
 'hyp:tok:for': 131.0,
'''


0.0%Pruned 22.44744744744745
{'pre:tag:nn': 511.0, 'hyp:tag:nn': 337.0, 'oth:overlap:overlap25': 291.0, 'hyp:tok:outside': 182.0, 'pre:tag:jj': 171.0, 'oth:overlap:overlap50': 166.0, 'hyp:tag:in': 163.0, 'pre:tag:in': 152.0, 'hyp:tag:dt': 149.0, 'hyp:tok:sleeping': 129.0, 'hyp:tok:outdoors': 124.0, 'hyp:tok:to': 122.0, 'pre:tag:dt': 121.0, 'pre:tag:.': 120.0, 'oth:overlap:overlap75': 120.0, 'hyp:tok:for': 119.0, 'pre:tok:man': 119.0, 'hyp:tok:there': 102.0, 'hyp:tok:people': 95.0, 'hyp:tok:eating': 90.0, 'hyp:tag:prp$': 87.0, 'hyp:tok:swimming': 86.0, 'hyp:tag:vbg': 75.0, 'hyp:tag:ex': 73.0, 'hyp:tok:sitting': 72.0, 'hyp:tok:tall': 70.0, 'pre:tok:woman': 69.0, 'hyp:tok:woman': 68.0, 'pre:tok:black': 66.0, 'hyp:tok:man': 66.0, 'pre:tok:blue': 60.0, 'pre:tok:dog': 59.0, 'pre:tok:sitting': 59.0, 'hyp:tok:cat': 51.0, 'pre:tok:girl': 50.0, 'hyp:tag:.': 48.0, 'hyp:tok:nobody': 48.0, 'hyp:tok:running': 47.0, 'pre:tok:red': 47.0, 'pre:tok:walking': 45.0, 'hyp:tok:human': 45.0, 'hyp:tok:inside'

"\n'pre:tag:nn': 539.0,\n 'hyp:tag:nn': 327.0,\n 'oth:overlap:overlap25': 307.0,\n 'oth:overlap:overlap50': 202.0,\n 'pre:tag:jj': 178.0,\n 'hyp:tok:outside': 149.0,\n 'hyp:tok:sleeping': 145.0,\n 'hyp:tag:in': 134.0,\n 'hyp:tag:vbg': 133.0,\n 'hyp:tag:dt': 133.0,\n 'pre:tag:.': 133.0,\n 'hyp:tok:for': 131.0,\n"

In [106]:
from collections import Counter
from itertools import chain

def get_nonfoundational_freq(neuron_formulas, foundationals):
    # flatten all concepts
    all_concepts = chain.from_iterable(neuron_formulas.values())

    # keep only non-foundationals
    non_foundationals = [c for c in all_concepts if c not in foundationals]

    # count frequency
    return Counter(non_foundationals)
from collections import Counter
from itertools import chain

def get_all_concept_freq(neuron_formulas):
    a=[]
    for u, co in neuron_formulas.items():
        for c in co:
            a.append(c)
    
    return Counter(a)

In [107]:
def get_pi(model, method, run):
    """
    Collect sparsity values (the number before '%Pruned')
    from directory names inside:

    /workspace/CCE_NLI/{MODEL}/exp/{method}/{run}/Expls

    Returns:
        Sorted list of floats.
    """
    base_path = f"/workspace/CCE_NLI/{model.upper()}/exp/{method}/{run}/Expls"

    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Path not found: {base_path}")

    sparsities = []

    for name in os.listdir(base_path):
        match = re.match(r"([0-9.]+)%Pruned", name)
        if match:
            sparsities.append(float(match.group(1)))
    if 0.0 not in sparsities:
        sparsities.insert(0,0.0)
    return sorted(sparsities)

run='Run0.25_5'
pis = get_pi('BERT', 'lottery_ticket', run)
print(pis)
for i,pi in enumerate(pis):
    if pi == 0.0:
        s=[]
        neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned')
        for u in neuron_formulas:
            for i in neuron_formulas[u]:
                s.append(i)
        print("ubique ", len(s))
    else:
        neuron_formulas = get_all_cps_for_pi(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/{run}/Expls/{pi}%Pruned')
        
    mask = build_binary_mask(neuron_formulas, foundationals)
    top,f,_=get_topk_concepts(mask,len(foundationals), foundationals)
    print(f"Avg number of times a foundational concept is repeated at sparsity {pi}%: {f}")
    nonfound=get_nonfoundational_freq(neuron_formulas, foundationals)
    print(f"sum of nonfound frewq : {sum(list(nonfound.values()))}. num found: {len(nonfound)}, percent:{sum(list(nonfound.values()))/len(nonfound)}")
    
    all_freq = get_all_concept_freq(neuron_formulas)
    total = sum(all_freq.values())
    unique = len(all_freq)

    print(
        f"All concepts total freq: {total}, "
        f"unique: {unique}, "
        f"avg repetition: {total / unique if unique else 0}"
    )


[0.0, 25.0, 43.75, 57.812, 68.359, 76.27]
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster1IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster1IOUS1024N.csv 2183
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster2IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster2IOUS1024N.csv 4295
IN  /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned /workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster3IOUS1024N.csv 7279
ubique  7279
sum of freqs : 5881.0. num found: 151
Avg number of times a foundational concept is repeated at sparsity 0.0%: 38.94701986754967
sum of nonfound frewq : 139

In [91]:
for i,pi in enumerate(['0.0', '25.0', '43.75', '57.812', '68.359', '76.27']):
    gen=0
    unit, unique1 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster1IOUS1024N.csv')
    unit, unique2 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster2IOUS1024N.csv')
    unit, unique3 = load_csv_data(f'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5/Expls/{pi}%Pruned/Cluster3IOUS1024N.csv')
    unique=unique1|unique2|unique3
    for cp in unique:
        if any(kw in cp for kw in [':tag:', 'oth:overlap']):
            gen += 1
    print(pi, gen/len(unique), gen, len(unique))

0.0 0.09368635437881874 46 491
25.0 0.09484536082474226 46 485
43.75 0.09270216962524655 47 507
57.812 0.09021113243761997 47 521
68.359 0.09325396825396826 47 504
76.27 0.0874751491053678 44 503


In [66]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.045283018867924

In [74]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/57.812%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.026415094339622

In [75]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/68.359%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.713207547169812

In [76]:
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/76.27%Pruned')
mask = build_binary_mask(neuron_formulas, foundationals)

top,f=get_topk_concepts(mask,len(foundationals), foundationals)
f

25.00377358490566

In [71]:
def get_non_foundationals(foundational, pi):
    unit_to_cp_dict = get_all_cps_for_pi(pi)
    allcps = set()
    for unit,cps in unit_to_cp_dict.items():
        allcps.update(cps)
    return list(allcps - set(foundational))
def get_topk_concepts(mask,k, concept_list):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    sorted_idx = np.argsort(col_sums)[::-1]
    top=[]
    freqs={}
    for i in range(len(sorted_idx))[:k]:
        idx = sorted_idx[i]
        top.append(concept_list[idx])
        freqs[concept_list[idx]]=col_sums[idx]
    return top, sum(list(freqs.values()))/len(freqs)
root_dir='/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls'
foundationals = get_foundationals(root_dir)
get_non_foundationals(foundationals,'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned' )
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned')
mask = build_binary_mask(neuron_formulas, non_foundationals)
_,f=get_topk_concepts(mask,len(non_foundationals), non_foundationals)
f

0.7145390070921985

In [70]:
get_non_foundationals(foundationals,'/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned' )
neuron_formulas = get_all_cps_for_pi('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/43.75%Pruned')
mask = build_binary_mask(neuron_formulas, non_foundationals)
_,f=get_topk_concepts(mask,len(non_foundationals), non_foundationals)
f

0.6968085106382979